# Restrudel — Phase 0: Setup

Mounts Google Drive, creates the dataset layout, installs Node + the Strudel
packages, and runs a sanity check that we can evaluate a Strudel pattern and
query its note events in Node.

**Run this in Google Colab** (connect via the VS Code Colab extension or open in
colab.research.google.com). Run cells top to bottom.

## 1. Mount Google Drive
Authorize when prompted. All data lives under Drive — nothing persists on the
Colab VM (which is ephemeral) or on your local machine.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Create the dataset layout on Drive
Single root `Restrudel/` matching roadmap.md Phase 0.

In [ ]:
import os

ROOT = '/content/drive/MyDrive/Restrudel'
SUBDIRS = [
    'corpus',                # raw fetched Strudel .js songs (Phase 1)
    'analysis',              # distribution tables + plots (Phase 1)
    'dataset/audio',         # {id}.wav
    'dataset/midi',          # {id}.mid
    'dataset/events',        # {id}.json (haps + synth params)
    'dataset/code',          # {id}.js (source pattern)
]
for d in SUBDIRS:
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)

print('Dataset root:', ROOT)
for d in sorted(SUBDIRS):
    print('  ', d)

## 3. Install Node.js
Colab is Ubuntu; the Strudel pattern engine is pure JS, so we run it in Node.
(Installing Node from a Python notebook is a little unusual but works fine.)

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs > /dev/null 2>&1
!node --version && npm --version

## 4. Install Strudel packages
Installed into a working dir on the VM (not Drive — node_modules on Drive is
painfully slow). Versions are pinned for reproducibility; bump deliberately.

In [ ]:
%%bash
mkdir -p /content/restrudel_node && cd /content/restrudel_node
npm init -y > /dev/null 2>&1
# TODO: pin exact versions once the eval import surface is verified (Phase 2)
npm install @strudel/core @strudel/mini @strudel/transpiler @tonejs/midi 2>&1 | tail -3
echo '--- installed ---'
node -e "console.log(require('@strudel/core/package.json').version)"

## 5. Sanity check — evaluate a pattern and query its events
Proves the Part A label path works end to end before we build anything. Expect
4 onsets (c3 eb3 g3 bb3) over 1 cycle with their begin/end times and synth params.

> Note: the exact import (`evaluate` from `@strudel/transpiler`) may differ by
> Strudel version. If this errors, that's the first thing to fix in Phase 2 —
> it tells us the real API surface of the installed version.

In [ ]:
%%writefile /content/restrudel_node/sanity.mjs
import { evaluate } from '@strudel/transpiler';
import '@strudel/mini';

const code = 'note("c3 eb3 g3 bb3").s("sawtooth").lpf(800)';
const { pattern } = await evaluate(code);
const haps = pattern.queryArc(0, 1).filter(h => h.hasOnset());
for (const h of haps) {
  console.log({
    note: h.value.note ?? h.value.n,
    begin: h.whole.begin.valueOf(),
    end: h.whole.end.valueOf(),
    s: h.value.s, lpf: h.value.lpf,
  });
}
console.log('onset count:', haps.length);

In [ ]:
!cd /content/restrudel_node && node sanity.mjs

## Done
If cell 5 printed 4 onsets, Phase 0 is complete and the Part A path is viable.
Next: **Phase 1** — fetch the Strudel song corpus into `Restrudel/corpus/` and
analyze the sound/synth distribution (`notebooks/01_strudel_corpus_analysis.ipynb`).